In [1]:
# Import libraries
import torch
import os
import wandb
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel, TaskType
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset, DatasetDict
from lib import full_cleanup

# Reset peak memory
def reset_peak_mem():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def report_peak_mem(tag: str = ""):
    if torch.cuda.is_available():
        print(f"Peak training memory{(' ' + tag) if tag else ''}: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

print("All imports successful!")

print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

print("Type of compute:", torch.cuda.get_device_name(0))

All imports successful!
Transformers version: 5.0.0
Torch version: 2.11.0a0+rocm7.12.0a20260203
CUDA available: True
Type of compute: Radeon 8060S Graphics


In [2]:
import os
from huggingface_hub import login

# Fetch the token from the environment securely
hf_token = os.environ.get("HF_TOKEN")

if hf_token is None:
    raise ValueError("HF_TOKEN environment variable not set.")

# Login using the fetched token
login(token=hf_token)

In [3]:
os.environ["WANDB_PROJECT"] = "spring_2026_research"
os.environ["WANDB_NOTEBOOK_NAME"] = "/home/andy/finetuning-workspace/workspace/gemma-lora.ipynb"
os.environ["WANDB_LOG_MODEL"] = "false"

os.environ["WANDB_WATCH"] = "false"

wandb.login()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/andy/.netrc.
wandb: Currently logged in as: andy2639le (andy2639le-atos) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
#MODEL = "google/gemma-3-270m-it"
#MODEL = "google/gemma-3-1b-it"  # Default model
MODEL = "google/gemma-3-4b-it"
#MODEL = "google/gemma-3-12b-it"
#MODEL = "google/gemma-3-27b-it"

#MODEL = "Qwen/Qwen3-VL-2B-Instruct"
#MODEL = "qwen/Qwen3-4B-Instruct-2507"

model_name = MODEL.split("/")[-1]

In [16]:
LR = 3e-5
EPOCHS = 1
BATCH_SIZE = 4
run_name = "gemma3-4b-lora-v1"

In [17]:
ds = load_dataset("itsanmolgupta/mimic-cxr-dataset")

print(ds)

def has_impression(example):
    return example["impression"] is not None and len(example["impression"].strip()) > 0

filtered_ds = ds["train"].filter(has_impression)
filtered_ds = filtered_ds.shuffle(seed=42).select(range(10000))

print("Original size:", len(ds["train"]))
print("Filtered size:", len(filtered_ds))

from datasets import DatasetDict

# Split the filtered dataset into train and temporary (test + validation)
# 80% Train, 20% Temp
train_test_valid_ds = filtered_ds.train_test_split(test_size=0.2, seed=42)

# Split the temporary set into validation and test (50% each of the 20% -> 10% of total each)
test_valid_ds = train_test_valid_ds['test'].train_test_split(test_size=0.5, seed=42)

# Combine into a single DatasetDict
split_ds = DatasetDict({
    'train': train_test_valid_ds['train'],
    'validation': test_valid_ds['train'],
    'test': test_valid_ds['test']
})

print("Dataset structure after splitting:")
print(split_ds)

DatasetDict({
    train: Dataset({
        features: ['image', 'findings', 'impression'],
        num_rows: 30633
    })
})
Original size: 30633
Filtered size: 10000
Dataset structure after splitting:
DatasetDict({
    train: Dataset({
        features: ['image', 'findings', 'impression'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['image', 'findings', 'impression'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['image', 'findings', 'impression'],
        num_rows: 1000
    })
})


In [18]:
from PIL import Image
import io

def collate_fn(examples):
    texts = []
    images = []
    
    for example in examples:
        img = example["images"][0]
        
        # Handle case where dataset serialized PIL Image to dict/bytes
        if isinstance(img, dict):
            img = Image.fromarray(img["array"]) if "array" in img else Image.open(io.BytesIO(img["bytes"]))
        elif isinstance(img, bytes):
            img = Image.open(io.BytesIO(img))
        
        # Ensure RGB (some CXR images are grayscale/RGBA)
        img = img.convert("RGB")
        images.append(img)
        
        text = processor.tokenizer.apply_chat_template(
            example["messages"], 
            tokenize=False, 
            add_generation_prompt=False
        )
        texts.append(text)
    
    batch = processor(
        text=texts, 
        images=[[img] for img in images], 
        return_tensors="pt", 
        padding=True
    )
    
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    
    return batch

In [19]:
def gemma3_format(ex):
    return {
        "images": [ex["image"]],   # top-level list of PIL images
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image"},   # placeholder only, no "image" key
                    {"type": "text",
                     "text": (
                         "You are assisting a radiologist. "
                         "Based on the given chest X-ray, generate a concise clinical impression."
                     )}
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": f"Impression: {ex['impression']}"}],
            },
        ]
    }

# Apply the formatting function to the dataset
# .map makes it so that it efficiently applies the format to dataset
# remove_columns so that the original columns (image, findings, and impression) are discarded and only the formatted data is left
formatted_ds = split_ds.map(gemma3_format, remove_columns=['image', 'findings', 'impression'])

# Verify the format
print("First example from train set:")
print(formatted_ds['train'][0]['messages'])

First example from train set:
[{'content': [{'text': None, 'type': 'image'}, {'text': 'You are assisting a radiologist. Based on the given chest X-ray, generate a concise clinical impression.', 'type': 'text'}], 'role': 'user'}, {'content': [{'text': 'Impression: Significant decrease in extent of right pleural effusion.', 'type': 'text'}], 'role': 'assistant'}]


In [20]:
model = AutoModelForImageTextToText.from_pretrained(
    MODEL, dtype="auto", device_map="auto", attn_implementation="eager"
)
processor = AutoProcessor.from_pretrained(MODEL, use_fast=False) # Changed from AutoTokenizer to AutoProcessor
torch_dtype = model.dtype

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"Weights footprint: {model.get_memory_footprint()/1e9:.2f} GB")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

trainable params: 11,898,880 || all params: 4,311,978,352 || trainable%: 0.2759
Weights footprint: 8.65 GB


In [21]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map="auto", attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"Weights footprint: {model.get_memory_footprint()/1e9:.2f} GB")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/opt/venv/lib64/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 11,898,880 || all params: 4,311,978,352 || trainable%: 0.2759
Weights footprint: 4.58 GB


In [22]:
args = SFTConfig(
    output_dir=f"output-{model_name}-lora",
    max_length=512,
    packing=False,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_checkpointing=False,
    optim="adamw_torch_fused",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=LR,
    fp16=True if torch_dtype == torch.float16 else False,
    bf16=True if torch_dtype == torch.bfloat16 else False,
    lr_scheduler_type="constant",
    report_to="wandb",
    run_name=run_name,
    remove_unused_columns=False, #added
    dataset_kwargs={"add_special_tokens": False, "append_concat_token": True},
    save_total_limit=1,  # keep only latest checkpoint
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=formatted_ds['train'],
    eval_dataset=formatted_ds['validation'],
    data_collator=collate_fn
)


In [12]:
reset_peak_mem()
trainer.train()
report_peak_mem("lora")
trainer.save_model()

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


/opt/venv/lib64/python3.13/site-packages/torch/_dynamo/eval_frame.py:1227: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.271263,2.266919


Peak training memory lora: 27.43 GB


In [12]:
# Free all memory
full_cleanup(model, trainer)

In [24]:
import glob
import os
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# ==========================================
# 1. FIND AND SELECT CHECKPOINT
# ==========================================
# Matches anything containing 'lora' or 'qlora' in the name
lora_checkpoints = sorted([d for d in glob.glob("output-*") if "full" not in d and ("lora" in d or "qlora" in d)])

if not lora_checkpoints:
    raise FileNotFoundError("No LoRA/QLoRA directories found (looking for 'output-*' excluding 'full').")

print(f"Found {len(lora_checkpoints)} adapter checkpoints:")
for i, ckpt in enumerate(lora_checkpoints):
    print(f"[{i}] {ckpt}")

selection = input(f"\nSelect checkpoint index [0-{len(lora_checkpoints)-1}] (default 0): ").strip()
try:
    idx = int(selection) if selection else 0
    selected_path = lora_checkpoints[idx]
except (ValueError, IndexError):
    print(f"Invalid selection '{selection}', defaulting to {lora_checkpoints[0]}")
    selected_path = lora_checkpoints[0]

print(f"\nLoading base model: {MODEL}")
print(f"Loading adapters from: {selected_path}")

# ==========================================
# 2. LOAD VLM BASE MODEL & PROCESSOR
# ==========================================
# Prevent the HIP Error crash by using the same 4-bit config from training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Use ImageTextToText and eager attention!
base = AutoModelForImageTextToText.from_pretrained(
    MODEL, 
    quantization_config=bnb_config,
    device_map="auto", 
    attn_implementation="eager" 
)

processor = AutoProcessor.from_pretrained(MODEL)
model = PeftModel.from_pretrained(base, selected_path)
model.eval()

# ==========================================
# 3. PREPARE MULTIMODAL INFERENCE
# ==========================================
# Extract the sample from your dataset
sample = formatted_ds['test'][0]

# Extract the image tensor directly from the dataset
# (Ensure 'image' matches the actual key in your dataset)
image = sample['images'] 

# Apply the chat template to bypass safety filters
prompt = processor.apply_chat_template(
    sample["messages"][:1], # Grabs only the User prompt portion
    tokenize=False, 
    add_generation_prompt=True
)

# Process both the image and the formatted text prompt
inputs = processor(
    images=image,
    text=prompt,
    return_tensors="pt"
).to(model.device)

# ==========================================
# 4. GENERATE AND DECODE
# ==========================================
print("\nGenerating impression...")
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=100)

# Slice off the prompt to isolate the generated text
input_length = inputs["input_ids"].shape[1]
generated_text = processor.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()

# ==========================================
# 5. DISPLAY RESULTS
# ==========================================
print(f"\nUser: {sample['messages'][0]['content']}")
print(f"\nExpected: {sample['messages'][1]['content']}")
print(f"\nGenerated: {generated_text}")

Found 1 adapter checkpoints:
[0] output-gemma-3-4b-it-lora



Select checkpoint index [0-0] (default 0):  0



Loading base model: google/gemma-3-4b-it
Loading adapters from: output-gemma-3-4b-it-lora


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


Generating impression...


/opt/venv/lib64/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



User: [{'text': None, 'type': 'image'}, {'text': 'You are assisting a radiologist. Based on the given chest X-ray, generate a concise clinical impression.', 'type': 'text'}]

Expected: [{'text': 'Impression: Complete left lower lobe collapse is potentially due to mucous plugging, but fixed endobronchial lesion is also possible such as carcinoid or foreign body. D. by , M. D.', 'type': 'text'}]

Generated: Impression: Mild interstitial edema. Unchanged interval findings, particularly for an appropriate small right pleural effusion. Small left pleural effusion. Interval stable cardiomegaly.


In [26]:
import torch
import evaluate
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

print("Loading base model and merging LoRA weights...")

BASE_MODEL_ID = "google/gemma-3-4b-it" 
LORA_WEIGHTS_DIR = "output-gemma-3-4b-it-lora" 

# 1. Bring back the 4-bit config to prevent the HIP error!
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

# Added bnb_config and eager attention
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID, 
    quantization_config=bnb_config,
    device_map="auto", 
    attn_implementation="eager"
)

# 2. Actually create the 'model' variable by applying the LoRA weights
model = PeftModel.from_pretrained(base_model, LORA_WEIGHTS_DIR)

# 1. Load the evaluation metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

predictions = []
references = []

print("Evaluating model on MIMIC-CXR test set...")
model.eval()

with torch.no_grad():
    for item in tqdm(formatted_ds["test"]):
        # Note: Changed 'images' to 'image' based on your previous working script. 
        # Double-check your dataset keys!
        image = item["images"]
        
        # 3. Extract the actual text string for the ground truth 
        # (Assuming the assistant's answer is the second item in the messages list)
        ground_truth = item["messages"][1]["content"] 

        # Format the prompt correctly to bypass the safety disclaimer
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": "You are assisting a radiologist. Based on the given chest X-ray, generate a concise clinical impression."}
                ]
            }
        ]
        
        prompt = processor.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )

        inputs = processor(
            images=image, 
            text=prompt, 
            return_tensors="pt"
        ).to(model.device)

        # Generate the clinical impression
        output_ids = model.generate(**inputs, max_new_tokens=128)

        # Decode and slice off the prompt so only the new text is evaluated
        input_length = inputs["input_ids"].shape[1]
        generated_text = processor.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()

        # Store for metric calculation
        predictions.append(generated_text)
        
        # BLEU expects references as a list of lists 
        references.append([ground_truth])

# Compute and print the final scores
print("\nCalculating Metrics...")

bleu_results = bleu.compute(predictions=predictions, references=references)
print(f"BLEU Score: {bleu_results['bleu']:.4f}")

rouge_results = rouge.compute(predictions=predictions, references=references)
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")

Loading base model and merging LoRA weights...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Evaluating model on MIMIC-CXR test set...


100%|███████████████████████████████████████████████████████████████████████████████| 1000/1000 [1:02:11<00:00,  3.73s/it]



Calculating Metrics...
BLEU Score: 0.0066
ROUGE-1: 0.1496
ROUGE-2: 0.0231
ROUGE-L: 0.1360
